In [1]:
# 评估tecent
from official_api.tencent import face_compare
import torch
from tqdm import tqdm
import os
import numpy as np

ths=[40,50,60]
result = {}
model_name="ArcFace"
with torch.no_grad():
    # 设置对抗样本目录路径 每月免费10000次调用，测三个刚好9000次，换个免费的继续测
    adv_samples_dirs = [
        f"data/FGSM_{model_name}_lfw_eps6_tpert4.4",
        f"data/MIM_{model_name}_lfw_eps6_tpert4.4",
        f"data/CW_{model_name}_lfw_eps16_tpert1",
        f"data/AT3D_{model_name}_eye_nose_lfw_eps5_tpert13.5",
        f"data/SiblingAttack_{model_name}_lfw_eps0.15_tpert10.7",
        f"data/AdvFace_lfw_eps8_tpert5.7",
        f"data/AdvMakeUP_lfw_tpert5.2",
        f"data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6",
    ]
    for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
        print("-----------------start white evaluate target method {0}-------------------\n".format(adv_samples_dir))
        save_path=f'apiresult/tencent/{adv_samples_dir[5:]}.npy'
        if not os.path.exists(save_path):
            result[adv_samples_dir]= np.empty((0, 3), float)
            # 列出目录中的所有文件和文件夹
            all_files_and_dirs = os.listdir(adv_samples_dir)
            # 过滤出所有子文件夹
            subdirectories = [d for d in all_files_and_dirs if os.path.isdir(os.path.join(adv_samples_dir, d))]
            for adv_pair in tqdm(subdirectories):
                base_dir = os.path.join(adv_samples_dir,adv_pair)
                adv_path = base_dir+"/adv.png"
                source_path = base_dir+"/source.png"
                target_path = base_dir+"/target.png"
                base_res = face_compare(face1_path=source_path, face2_path=target_path)
                FSS_res = face_compare(face1_path=adv_path, face2_path=source_path)
                FTS_res = face_compare(face1_path=adv_path, face2_path=target_path)
                if base_res is not None and FSS_res is not None and FTS_res is not None:
                    result[adv_samples_dir] = np.vstack([result[adv_samples_dir], [base_res,FSS_res,FTS_res]])
            print("tencent"+adv_samples_dir+"上失败了："+str(1000-len(result[adv_samples_dir])))
            np.save(save_path, np.array(result[adv_samples_dir]))
        else:
            result[adv_samples_dir] = np.load(save_path)
        num = len(result[adv_samples_dir])
        print("有效数据个数 ",num)
        for th in ths:
            print(th, np.sum(result[adv_samples_dir][:,0] > th)/num, np.sum(result[adv_samples_dir][:,2] > th)/num, np.sum((result[adv_samples_dir][:,1] > th) & (result[adv_samples_dir][:,2] > th))/num)

-----------------start white evaluate target method data/FGSM_ArcFace_lfw_eps6_tpert4.4-------------------

有效数据个数  998
40 0.01603206412825651 0.40180360721442887 0.40180360721442887
50 0.006012024048096192 0.21142284569138275 0.21142284569138275
60 0.001002004008016032 0.08917835671342686 0.08917835671342686
-----------------start white evaluate target method data/MIM_ArcFace_lfw_eps6_tpert4.4-------------------

有效数据个数  1000
40 0.016 0.673 0.67
50 0.006 0.512 0.501
60 0.001 0.359 0.339
-----------------start white evaluate target method data/CW_ArcFace_lfw_eps16_tpert1-------------------

有效数据个数  1000
40 0.016 0.061 0.061
50 0.006 0.017 0.017
60 0.001 0.003 0.003
-----------------start white evaluate target method data/AT3D_ArcFace_eye_nose_lfw_eps5_tpert13.5-------------------

有效数据个数  1000
40 0.017 0.687 0.534
50 0.004 0.416 0.252
60 0.001 0.206 0.082
-----------------start white evaluate target method data/SiblingAttack_ArcFace_lfw_eps0.15_tpert10.7-------------------

有效数据个数  100

In [2]:
print("tencent 0.01%FAR ASR1:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum(result[adv_samples_dir][:,2] > ths[1])/num*100:.1f}")
print("tencent 0.01%FAR ASR2:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum((result[adv_samples_dir][:, 1] > ths[1]) & (result[adv_samples_dir][:, 2] > ths[1])) / num*100:.1f}")

tencent 0.01%FAR ASR1:
21.1
51.2
1.7
41.6
85.2
41.1
4.9
76.5
tencent 0.01%FAR ASR2:
21.1
50.1
1.7
25.2
39.5
29.3
4.9
58.4


In [3]:
print("tencent 0.1%FAR ASR1:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum(result[adv_samples_dir][:,2] > ths[0])/num*100:.1f}")
print("tencent 0.1%FAR ASR2:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum((result[adv_samples_dir][:, 1] > ths[0]) & (result[adv_samples_dir][:, 2] > ths[0])) / num*100:.1f}")

tencent 0.1%FAR ASR1:
40.2
67.3
6.1
68.7
92.7
59.2
15.8
88.2
tencent 0.1%FAR ASR2:
40.2
67.0
6.1
53.4
60.5
50.6
15.8
80.3
